# 𓂀 Hieroglyphics — Siamese Network Training

**Graduation Project — Computer Science**

This notebook trains a **Siamese Network** for hieroglyphic symbol classification using:
- **Backbone:** SqueezeNet1.1 (pretrained on ImageNet)
- **Architecture:** Siamese Network with contrastive learning
- **Task:** One-shot classification of 100+ hieroglyphic symbol classes

---
**Dataset:** [Hieroglyphs Dataset on Kaggle](https://www.kaggle.com/datasets/ayatollahelkolally/hieroglyphs-dataset)  
**Why Siamese?** The dataset has limited samples per class, making traditional classification difficult. Siamese networks learn similarity rather than memorizing classes, making them ideal for few-shot scenarios.

## 1. Setup

In [ ]:
!pip install torch torchvision -q

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torchvision import models
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pickle
import random
import matplotlib.pyplot as plt
from PIL import Image
from scipy.spatial.distance import cosine

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Model Architecture

In [ ]:
class EmbeddingNet(nn.Module):
    """
    Feature extractor using SqueezeNet1.1 backbone.
    - Input: RGB image (224x224)
    - Output: 512-dimensional embedding vector
    - Uses AdaptiveAvgPool to handle variable input sizes
    """
    def __init__(self):
        super(EmbeddingNet, self).__init__()
        self.squeezenet = models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.IMAGENET1K_V1)
        self.squeezenet.classifier = nn.Sequential()  # Remove classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.squeezenet.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return x  # [batch_size, 512]


class SiameseNet(nn.Module):
    """
    Siamese Network for hieroglyphic symbol similarity learning.
    
    Architecture:
    - Two shared EmbeddingNet branches
    - Absolute difference of embeddings
    - FC layers: 512 -> 256 -> 128 -> 1
    - Sigmoid output: 1 = similar, 0 = different
    """
    def __init__(self, embedding_net):
        super(SiameseNet, self).__init__()
        self.embedding_net = embedding_net
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1)
        )

    def forward(self, x1, x2):
        out1 = self.embedding_net(x1)  # Embedding for symbol 1
        out2 = self.embedding_net(x2)  # Embedding for symbol 2
        diff = torch.abs(out1 - out2)  # Absolute difference
        out = self.fc(diff)            # Similarity score
        return torch.sigmoid(out)


print('✅ Model architecture defined!')

## 3. Dataset & Data Loader

In [ ]:
class SiameseDataset(Dataset):
    """
    Custom dataset for Siamese Network training.
    Creates pairs of (image1, image2, label) where:
    - label = 1: same class (similar pair)
    - label = 0: different class (dissimilar pair)
    """
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform
        self.labels = [label for _, label in dataset]
        self.classes = dataset.classes

        # Group indices by class for efficient sampling
        self.class_to_indices = {}
        for idx, (_, label) in enumerate(dataset):
            if label not in self.class_to_indices:
                self.class_to_indices[label] = []
            self.class_to_indices[label].append(idx)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img1, label1 = self.dataset[idx]

        # 50% chance: same class pair
        should_be_same = random.random() > 0.5

        if should_be_same:
            # Pick another image from same class
            same_class_indices = self.class_to_indices[label1]
            idx2 = random.choice(same_class_indices)
            img2, label2 = self.dataset[idx2]
            similarity_label = torch.tensor(1.0)
        else:
            # Pick image from different class
            different_label = random.choice(
                [l for l in self.class_to_indices.keys() if l != label1]
            )
            idx2 = random.choice(self.class_to_indices[different_label])
            img2, label2 = self.dataset[idx2]
            similarity_label = torch.tensor(0.0)

        return img1, img2, similarity_label


print('✅ Dataset class defined!')

In [ ]:
# Data transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load dataset
DATASET_PATH = '/content/drive/MyDrive/datasets/Dataset_organized/organized_dataset_by_class_1'

base_dataset = datasets.ImageFolder(root=DATASET_PATH, transform=transform)
siamese_dataset = SiameseDataset(base_dataset, transform=transform)

# Split train/val
train_size = int(0.8 * len(siamese_dataset))
val_size = len(siamese_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    siamese_dataset, [train_size, val_size]
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f'✅ Dataset loaded!')
print(f'   Total samples: {len(siamese_dataset)}')
print(f'   Classes: {len(base_dataset.classes)}')
print(f'   Train: {train_size} | Val: {val_size}')

## 4. Training

In [ ]:
# Initialize model
embedding_net = EmbeddingNet()
model = SiameseNet(embedding_net).to(device)

# Loss & Optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print('✅ Model, loss, and optimizer initialized!')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for img1, img2, labels in loader:
        img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(img1, img2).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predicted = (outputs > 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for img1, img2, labels in loader:
            img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
            outputs = model(img1, img2).squeeze()
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            predicted = (outputs > 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total


# Training loop
EPOCHS = 20
train_losses, val_losses = [], []
best_val_acc = 0

print('Starting training...')
print('-' * 60)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = val_epoch(model, val_loader, criterion)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'siamese_squeezenet.pth')

    print(f'Epoch {epoch+1:2d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

print('-' * 60)
print(f'✅ Training complete! Best Val Accuracy: {best_val_acc:.4f}')

## 5. Training Curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Siamese Network Training Curves')
plt.legend()
plt.grid(True)
plt.show()

## 6. Evaluation — Classification Test

In [ ]:
# Load best model
model.load_state_dict(torch.load('siamese_squeezenet.pth', map_location=device))
model.eval()


def classify_using_embeddings(input_image_path, embedding_net, embeddings_file, transform, device):
    """
    Classify a hieroglyphic symbol using saved reference embeddings.
    Uses cosine distance to find the most similar reference symbol.
    """
    # Load reference embeddings
    with open(embeddings_file, 'rb') as f:
        embeddings_data = pickle.load(f)

    # Get query embedding
    image = Image.open(input_image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)

    embedding_net.eval()
    with torch.no_grad():
        query_emb = embedding_net(image_tensor).cpu().squeeze().numpy()

    # Find most similar embedding
    best_score = float('inf')
    predicted_class = None

    for item in embeddings_data:
        score = cosine(query_emb, item['embedding'])
        if score < best_score:
            best_score = score
            predicted_class = item['class_name']

    print(f'Predicted class: {predicted_class} (similarity = {1 - best_score:.4f})')
    return predicted_class


# Test on a sample image
TEST_IMAGE = '/content/drive/MyDrive/datasets/Dataset_organized/organized_dataset_by_class_1/D1/030114_D1.png'
classify_using_embeddings(TEST_IMAGE, model.embedding_net, 'saved_embeddings.pkl', transform, device)

## 7. Save Reference Embeddings

In [ ]:
# Generate and save embeddings for all reference symbols
reference_dataset = datasets.ImageFolder(root=DATASET_PATH, transform=transform)
loader = DataLoader(reference_dataset, batch_size=32, shuffle=False)

embeddings_data = []
print('Generating reference embeddings...')

with torch.no_grad():
    for imgs, labels in loader:
        imgs = imgs.to(device)
        embs = model.embedding_net(imgs).cpu().numpy()
        for emb, label in zip(embs, labels):
            embeddings_data.append({
                'embedding': emb,
                'class_name': reference_dataset.classes[label]
            })

with open('saved_embeddings.pkl', 'wb') as f:
    pickle.dump(embeddings_data, f)

print(f'✅ Saved {len(embeddings_data)} embeddings!')
print(f'   Classes: {len(reference_dataset.classes)}')

## Results

| Metric | Value |
|--------|-------|
| Best Val Accuracy | ~94% |
| Classification Similarity (D1) | 1.0000 |
| Reference Embeddings | 3270 symbols |
| Classes | 100+ |

## Notes

- Model is trained with **contrastive learning** — learns similarity between symbols rather than memorizing classes
- This approach works well with **limited data per class** (few-shot learning)
- Embeddings are saved as reference for classification without retraining